In [16]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [17]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [18]:
data = pd.read_csv('data/grade_b.csv')

In [19]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611126...,6111266962187,Lait – Jaouda – 450 ml,450 ml,NaN,Jaouda,"Dairies, ,, Meals, ,, Milks (liquid and powder...",NaN,NaN,NaN,...,-3,4,7,3.0,0,0.0,1,7.0,0,0
1,https://world.openfoodfacts.org/product/611124...,6111242101180,uht jaouda 1L – 1L,1L,"Multilayer-composite, ,, Tetra Pak",Jaouda,"Dairies, ,, Milks (liquid and powder), ,, Milk...",No gluten,NaN,NaN,...,-1,5,6,5.0,0,0.0,0,6.0,0,0
2,https://world.openfoodfacts.org/product/611103...,6111032009665,Salim – 470 ml,470 ml,NaN,Salim,"Dairies, ,, Milks (liquid and powder), ,, Milk...",No gluten,NaN,NaN,...,-3,3,6,3.0,0,0.0,0,6.0,0,0
3,https://world.openfoodfacts.org/product/611124...,6111242100206,Yaourt nature – Jaouda – 110 g,110 g,NaN,Jaouda,"Dairies, ,, Fermented foods, ,, Fermented milk...",NaN,NaN,Maroc,...,1,2,1,0.0,0,2.0,0,1.0,0,0
4,https://world.openfoodfacts.org/product/611125...,6111252421582,"اكوافينا – pepsi – 1,5l","1,5l",NaN,pepsi,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,0,0,0,0.0,0,0.0,0,0.0,0,0


In [ ]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)

In [21]:
data[["miktar","birim"]].head()

,miktar,birim
0,450.0,ml
1,1.0,l
2,470.0,ml
3,110.0,g
4,1.5,l


In [22]:
data.drop(columns=["url","urun_adi","kategoriler","etiketler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni"], inplace=True)

In [23]:
data["etiketler_listesi"].head(20)

0                                                    []
1                                           [No gluten]
2                                           [No gluten]
3                                                    []
4                                                    []
5     [No artificial flavors, No preservatives, Gree...
6     [Low or no salt, Organic, Vegetarian, Certifie...
7     [Vegetarian, Source of fibre, High fibres, No ...
8     [Low or no fat, Low or no sugar, Vegetarian, L...
9     [Low or no sugar, Vegetarian, Vegan, No added ...
10    [No gluten, Organic, Catalan Council of Organi...
11    [Low or no sugar, Low sugar, Enriched with vit...
12                                              [Halal]
13    [No preservatives, Source of fibre, High fibre...
14    [Low or no fat, Pasteurized, Source of protein...
15                                          [No gluten]
16    [Low or no sugar, Vegetarian, Source of fibre,...
17                                              

In [24]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (155): ['20002183', '20004132', '20171841', '20267605', '3019081241216', '3023290030547', '3029330004240', '3029330022428', '3029330069898', '3029330071280'] ...
miktar (56): ['0.5', '0.7', '1.0', '1.5', '100.0', '1000.0', '110.0', '116.0', '120.0', '130.0'] ...
markalar (102): ['ALHORRA', 'Albane', 'Alesto', 'Alpro', 'Aquafina', 'Baresa', 'Bjorg', 'Blini', 'Brets', 'Briau'] ...
alerjenler (28): ["['Celery', 'Gluten', 'Soybeans']", "['Celery', 'Gluten']", "['Celery']", "['Eggs', 'Gluten', 'Milk', 'Soybeans']", "['Fish', 'Milk', 'Sesame seeds']", "['Fish', 'Soybeans']", "['Fish', 'Sulphur dioxide and sulphites']", "['Fish']", "['Gluten', 'Milk', 'Soybeans']", "['Gluten', 'Milk', 'fr:avoine']"] ...
eser_miktarlar (51): ["['Celery', 'Crustaceans', 'Eggs', 'Fish', 'Gluten', 'Lupin', 'Milk', 'Molluscs', 'Mustard', 'Soybeans']", "['Celery', 'Crustaceans', 'Eggs', 'Fish', 'Mustard']", "['Celery', 'Crustaceans', 'Gluten', 'Molluscs', 'Mustard', 'Nuts', 'Peanuts', 'Sesame seeds', 'Soybea

In [25]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     128
False     24
Name: count, dtype: int64

In [26]:
data.isnull().mean() * 100

barkod                           0.000000
miktar                           3.870968
markalar                         1.290323
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                    0.000000
nutriscore_notu                  0.000000
nova_grubu                      11.612903
yesil_skor_notu                 12.258065
palmiye_yagi_icermez             1.935484
vejetaryen                      14.838710
vegan_durumu                     0.000000
yag_seviyesi                     0.645161
doymus_yag_seviyesi              0.645161
seker_seviyesi                   0.645161
tuz_seviyesi                     0.645161
enerji_kj                        0.000000
enerji_kcal                      0.000000
yag_g                            0.000000
doymus_yag_g                     0.000000
karbonhidrat_g                   1.290323
seker_g                          0.000000
lif_g                           27.741935
protein_g                        0

In [27]:
data.to_csv('data/cleaned_food_data.csv', index=False)

In [28]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'yag_seviyesi',
       'doymus_yag_seviyesi', 'seker_seviyesi', 'tuz_seviyesi', 'enerji_kj',
       'enerji_kcal', 'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g',
       'lif_g', 'protein_g', 'tuz_g', 'sodyum_g', 'alkol_yuzde',
       'meyve_sebze_baklagil_yuzde', 'nutriscore_puan', 'ns_negatif_puan',
       'ns_pozitif_puan', 'ns_enerji_puan', 'ns_seker_puan',
       'ns_doymus_yag_puan', 'ns_tuz_puan', 'ns_protein_puan', 'ns_lif_puan',
       'ns_meyve_sebze_baklagil_puan', 'urun_bilgisi', 'birim',
       'kategori_listesi', 'etiketler_listesi'],
      dtype='object')